### SECTION 1: Import Dependencies

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import *
import pandas as pd
from sqlalchemy import create_engine

### SECTION 2: Configuration

In [2]:
DATA_PATH = r"C:\Users\USER\Desktop\Data Recap\pyspark\etl_nuga_bank\rawdata\nuga_bank_transactions.csv"
APP_NAME = "NugaBankETL"
# POSTGRES_DB = "..."
# POSTGRES_USER = "..."

### SECTION 3: Spark Session

In [3]:
# Initialise the spark session
spark = (
    SparkSession.builder
    .appName('NugaBankETL')
    .master('local[*]')
    .getOrCreate())

### SECTION 4: Data Extraction / Loading the csv 

In [4]:
nuga_df = (
    spark.read
        .option("header", True) # this means use the first row as column names
        .option("inferSchema", True) # this scans the file, it notices numbers, so creates integerTyp or double, depending on datatype
        .csv(DATA_PATH)
)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/C:/Users/USER/Desktop/Data Recap/pyspark/etl_nuga_bank/rawdata/nuga_bank_transactions.csv.

### SECTION 5: Data Profiling

In [ ]:
# nuga_df.show(10, truncate=False)
# nuga_df.printSchema()
# nuga_df.columns
# nuga_df.count()
nuga_df.describe().show()


+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+------+---------+--------------------+------+--------------+
|summary|            Amount|Transaction_Type|Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|      Company|         Job_Title|              Email|        Phone_Number|  Credit_Card_Number|                IBAN|Currency_Code|     Random_Number|Category| Group|Is_Active|         Description|Gender|Marital_Status|
+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+------+---------+---------

In [ ]:
nuga_df.selectExpr(
    "count(*) as total_rows",
    "count(distinct Customer_Name) as unique_customers"
).show()

+----------+----------------+
|total_rows|unique_customers|
+----------+----------------+
|   1000000|          312447|
+----------+----------------+



In [ ]:
# Display how many null values each colum contains
# from pyspark.sql.functions import col, count, when

nuga_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in nuga_df.columns
]).show(vertical=True)

-RECORD 0--------------------
 Transaction_Date   | 0      
 Amount             | 0      
 Transaction_Type   | 0      
 Customer_Name      | 100425 
 Customer_Address   | 100087 
 Customer_City      | 100034 
 Customer_State     | 100009 
 Customer_Country   | 100672 
 Company            | 100295 
 Job_Title          | 99924  
 Email              | 100043 
 Phone_Number       | 100524 
 Credit_Card_Number | 100085 
 IBAN               | 100300 
 Currency_Code      | 99342  
 Random_Number      | 99913  
 Category           | 100332 
 Group              | 100209 
 Is_Active          | 100259 
 Last_Updated       | 100321 
 Description        | 100403 
 Gender             | 99767  
 Marital_Status     | 99904  



In [ ]:
# this shows us exactly which values exist in Is_Active e.g (Yes, No, Null, etc)
nuga_df.groupBy("Is_Active").count().show()

+---------+------+
|Is_Active| count|
+---------+------+
|     NULL|100259|
|       No|449899|
|      Yes|449842|
+---------+------+



In [ ]:
"""
this will tell us that the transaction types are consistent 
and don't contain issues such as different capitalization like:
(Deposit, deposit, DEPOSIT) or mispellings
""" 
nuga_df.groupBy("Transaction_Type").count().show()

+----------------+------+
|Transaction_Type| count|
+----------------+------+
|         Deposit|333120|
|        Transfer|333301|
|      Withdrawal|333579|
+----------------+------+



### SECTION 6: Data Cleaning

In [ ]:
# Step 1 - Import dependencies
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, count

def summarize_missing_values(df: DataFrame) -> DataFrame:
    """
    Generate a summary of missing (NULL) values for every column in a Spark DataFrame

    Parameter
    ---------
        df : DataFrame 
        Input Spark DataFrame to profile

    Returns
    -------
    DataFrame
        A single-row Spark DataFrame where each colum contains the number of NULL values 
        found in the corresponding input column
    """

    return df.select(
        [
            count(when(col(column).isNull(), column)).alias(column)
            for column in df.columns
        ]
    )


In [ ]:
def fill_missing_string_values(df: DataFrame) -> DataFrame:
    """
    Fill missing values for categorical text columns.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame containing transaction records

    
    Returns
    -------
    DataFrame
        Spark DataFrame with missing string value replaced
    """

    fill_values = {
        "Customer_Name" : "Unknown",
        "Customer_Address" : "Unknown",
        "Customer_City" : "Unknown",
        "Customer_State" : "Unknown",
        "Customer_country" : "Unknown",
        "Company" : "Unknown",
        "Job_Title" : "Unknown",
        "Email" : "Unknown",
        "Phone_Number" : "Unknown",
        "Category" : "Unknown",
        "Description" : "No Description",
        "Gender" : "Unknown",
        "Marital_Status" : "Unknown"
    }

    return df.fillna(fill_values)

In [ ]:
def standardize_column_names(
        df: DataFrame,
)-> DataFrame:
    """
    Standardise all columns names

    This function:
    1. Removes leading/trailing spaces.
    2. Converts names to lowercase.
    3. Replaces spaces with underscores.

    Parameters
    ----------
    df: DataFrame
        input Spark DataFrame

    Returns
    -------
    DataFrame
        DataFrame with standardized columns
    """

    for column in df.columns:

        new_column = (
            column
            .strip()
            .lower()
            .replace(" ", "_")
        )

        df = df.withColumnRenamed(
            column,
            new_column
        )
    
    return df



nuga_df = standardize_column_names(nuga_df)


nuga_df.columns


# ///////////////////////////////////--Let's Build the Production Version--//////////////////////////////////////////////
# import re

# def standardize_column_names(
#     df:DataFrame
# )-> DataFrame:
#     """
#     Standardize all DataFrame column names.

#     This function:
#     1. Removes leading and trailing whitespace.
#     2. Converts names to lowercase.
#     3. Replaces spaces with underscores.
#     4. Removes non-alphanumeric characters (except underscores).
#     5. Detects duplicate names created during standardization.

#     Parameters
#     ----------
#     df : DataFrame
#         Input Spark DataFrame.

#     Returns
#     -------
#     DataFrame
#         DataFrame with standardized column names.

#     Raises
#     ------
#     ValueError
#         If duplicate column names are produced.
#     """
        
#     standardized_columns = [
#         re.sub(r"[^a-z0-9_]", "", column.strip().lower().replace(" ", "_"))
#         for column in df.columns
#     ]


#     duplicates = {
#         column
#         for column in standardized_columns
#         if standardized_columns.count(column) > 1
#     }

#     if duplicates:
#         raise ValueError(
#             f"Duplicate column names after standardization: {sorted(duplicates)}"
#         )

#     return df.toDF(*standardized_columns)




['transaction_date',
 'amount',
 'transaction_type',
 'customer_name',
 'customer_address',
 'customer_city',
 'customer_state',
 'customer_country',
 'company',
 'job_title',
 'email',
 'phone_number',
 'credit_card_number',
 'iban',
 'currency_code',
 'random_number',
 'category',
 'group',
 'is_active',
 'last_updated',
 'description',
 'gender',
 'marital_status']

In [ ]:
# Verify the cleaning (A professional Engineer never assumes a transformation worked, we verify it.)
nuga_df = fill_missing_string_values(nuga_df)

summarize_missing_values(nuga_df).show(vertical=True)

-RECORD 0--------------------
 transaction_date   | 0      
 amount             | 0      
 transaction_type   | 0      
 customer_name      | 0      
 customer_address   | 0      
 customer_city      | 0      
 customer_state     | 0      
 customer_country   | 0      
 company            | 0      
 job_title          | 0      
 email              | 0      
 phone_number       | 0      
 credit_card_number | 100085 
 iban               | 100300 
 currency_code      | 99342  
 random_number      | 99913  
 category           | 0      
 group              | 100209 
 is_active          | 100259 
 last_updated       | 100321 
 description        | 0      
 gender             | 0      
 marital_status     | 0      



In [ ]:
# A better Way to Verify
# Instead of checking every column manually, lets create another reusable function

def display_dataframe_info(df: DataFrame) -> None:
    """
    Parameters
    ----------
    df: DataFrame
        Input Spark DataFrame


    Returns
    -------
    None
    """

    print("=" * 70)
    print("DataFrame Summary")
    print("=" * 70)

    print(f"Rows    : {df.count():,}")
    print(f"Columns : {len(df.columns)}")

    print("\nSchema")
    df.printSchema()


display_dataframe_info(nuga_df)

DataFrame Summary
Rows    : 1,000,000
Columns : 23

Schema
root
 |-- transaction_date: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- customer_address: string (nullable = false)
 |-- customer_city: string (nullable = false)
 |-- customer_state: string (nullable = false)
 |-- customer_country: string (nullable = false)
 |-- company: string (nullable = false)
 |-- job_title: string (nullable = false)
 |-- email: string (nullable = false)
 |-- phone_number: string (nullable = false)
 |-- credit_card_number: long (nullable = true)
 |-- iban: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- random_number: double (nullable = true)
 |-- category: string (nullable = false)
 |-- group: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- description: string (nullable = false)
 |-- gender: 

##### Drop Duplicate

In [ ]:
def count_duplicate_rows(df: DataFrame)->int:
    """
    Count the number of exact duplicates rows in a Spark DataFrame

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    Returns
    -------
    int
        Number of duplicate rows
    """

    total_rows = df.count()
    unique_rows = df.drop_duplicates().count()

    return total_rows - unique_rows

In [ ]:
duplicate_count = count_duplicate_rows(nuga_df)

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


In [ ]:
# Inspect the Duplicates
# Counting duplicates is useful, but seeing them is even more valuable.

def show_duplicates_rows(df: DataFrame, limit: int = 10) -> None:
    """
    Display a sample of duplicate rows.

    Parameters
    ----------
    df : DataFrame
    Input Spark DataFrame.


    limit : int, optional
        Maximum number of duplicates to display
        Defaults to 10.

    Returns
    -------
    None
    """

    (
        df.groupBy(df.columns)
        .count()
        .filter("count > 1")
        .show(limit, truncate=False)
    )

In [ ]:
show_duplicates_rows(nuga_df)

+----------------+------+----------------+-------------+----------------+-------------+--------------+----------------+-------+---------+-----+------------+------------------+----+-------------+-------------+--------+-----+---------+------------+-----------+------+--------------+-----+
|transaction_date|amount|transaction_type|customer_name|customer_address|customer_city|customer_state|customer_country|company|job_title|email|phone_number|credit_card_number|iban|currency_code|random_number|category|group|is_active|last_updated|description|gender|marital_status|count|
+----------------+------+----------------+-------------+----------------+-------------+--------------+----------------+-------+---------+-----+------------+------------------+----+-------------+-------------+--------+-----+---------+------------+-----------+------+--------------+-----+
+----------------+------+----------------+-------------+----------------+-------------+--------------+----------------+-------+---------+--

In [ ]:
# Data Type Standardization

# Before we change any data, let's inspect it.
nuga_df.groupBy("is_Active").count().show()

+---------+------+
|is_Active| count|
+---------+------+
|     NULL|100259|
|       No|449899|
|      Yes|449842|
+---------+------+



In [ ]:
def convert_yes_no_to_boolean(
        df: DataFrame,
        column_name: str
) -> DataFrame:
    """
    Convert a Yes/No string column to Boolean.

    Parameter
    ---------
    df : DataFrame
        input spark DataFrame.

    column_name: str
        Name of the column to convert.

    Returns
    -------
    DataFrame
        New DataFrame with the specified column converted
        to Boolean values
    """

    return df.withColumn(
        column_name,
        when(col(column_name) == "Yes", True)
        .when(col(column_name) == "No", False)
        .otherwise(None)
    )


nuga_df = convert_yes_no_to_boolean(
    nuga_df, "is_Active"
)

nuga_df.printSchema()



root
 |-- transaction_date: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- customer_address: string (nullable = false)
 |-- customer_city: string (nullable = false)
 |-- customer_state: string (nullable = false)
 |-- customer_country: string (nullable = false)
 |-- company: string (nullable = false)
 |-- job_title: string (nullable = false)
 |-- email: string (nullable = false)
 |-- phone_number: string (nullable = false)
 |-- credit_card_number: long (nullable = true)
 |-- iban: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- random_number: double (nullable = true)
 |-- category: string (nullable = false)
 |-- group: string (nullable = true)
 |-- is_Active: boolean (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- description: string (nullable = false)
 |-- gender: string (nullable = false)
 |-- marital_status: string (nul

In [ ]:
# Lets inspect the values

nuga_df.groupBy("is_Active").count().show()

+---------+------+
|is_Active| count|
+---------+------+
|     NULL|100259|
|     true|449842|
|    false|449899|
+---------+------+



#### N/B since there is no duplicate, no need to run the below syntax
nuga_df = nuga.dropDuplicates()

### Date and Timestamp Cleaning


##### Since inferschema already converted Transactio_Date from string to Timestamp which is correct, no need to do it again.

### SECTION 7: Data Normalisation / Transformation

In [ ]:
# Function to trim columns

from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

def trim_string_columns(df: DataFrame) -> DataFrame:
    """
    Remove leading and trailing whitespace from every
    string column in a spark DataFrame.

    Parameters
    ----------
    df: DataFrame
        input spark DataFrame.

    Returns
    -------
    DataFrame
        DataFrame with all string columns trimmed.
    """
    expressions = []

    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            expressions.append(
                trim(col(field.name)).alias(field.name)
            )
        else:
            expressions.append(
                col(field.name)
            )
        
    return df.select(*expressions)

nuga_df = trim_string_columns(nuga_df)

## Validation

In [ ]:
# validation 1: validate required columns

from typing import List

def validate_required_columns(
        df: DataFrame,
        required_columns: List[str]
)->None: # We are returning None because this function does not transform data, its job is to stop the pipeline if something is wrong.
    
    
    """
    Validate that all required columns exists in the DataFrame

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    required_columns : List[str]
        List of expected column names.


    Raises
    ------
    ValueError
        If one or more required columns are missing.
    """

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {', '.join(missing_columns)}"
        )
    





In [ ]:
# from pyspark.sql.types import NumericType
# def validate_numeric_column(
#     df: DataFrame,
#     column_name: str
# )-> bool:
    
#     """
#     Validate that a column has a numeric Spark data type.

#     Parameters
#     ----------
#     df : DataFrame
#         Input Spark DataFrame.

#     column_name : str
#         Name of the column to validate.

#     Returns
#     -------
#     bool
#         True if the column is numeric.

#     Raises
#     ------
#     ValueError
#         If the column does not exist.
#     """

#     if column_name not in df.columns:
#         raise ValueError(f"Column {column_name} does not exist")
    
#     data_type = df.schema[column_name].dataType

#     return isinstance(data_type, NumericType)

#     if validate_numeric_column(df, "amount"):
#         print("Proceed with numeric calculations.")
#     else:
#         print("Column must be converted first.")



In [ ]:
def validate_numeric_content(
        df: DataFrame,
        column_name: str
)-> None:
    
    """
    Check that all non-values in column are numeric using regex

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    Column_name: str
        name of column to validate

    Return:
    -------
       None

    Raises
    ------
    ValueError
        If column does not exists or contains non-numeric values
    """

    if column_name not in df.columns:
        raise ValueError(f"column '{column_name}' does not exist")
    
    NUMERIC_PATTERN  = r"^-?\d+(\.\d+)?$"

    invalid_rows = df.filter(
        col(column_name).isNotNull() &
        (~col(column_name).rlike(NUMERIC_PATTERN))
    )

    invalid_count = invalid_rows.count()

    if invalid_count > 0:
        raise ValueError(
            f"column '{column_name}' contains {invalid_count} non-numeric values(s)."
        )
    




In [ ]:
def validate_column_types(
    df: DataFrame,
    expected_schema: dict[str, DataType]
)-> None:
    """
    Validate that the DataFrame contains the expected columns and
    that each column has the expected Spark data type.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame.

    expected_schema : dict[str, DataType]
        Dictionary mapping column names to their expected Spark
        data types.

    Raises
    ------
    ValueError
        If one or more expected columns are missing or if one or
        more columns have an unexpected data type.

    Returns
    -------
    None
        Returns None if all columns exist and their data types
        match the expected schema.
    """
    validation_errors = []

    for column_name, expected_type in expected_schema.items():

        # check whether the column exists
        if column_name not in df.columns:
            validation_errors.append(
                f"- Missing column: '{column_name}'"
            )
            continue

        actual_type = df.schema[column_name].dataType

        # Compare Spark data types
        if type(actual_type) != type(expected_type):
            validation_errors.append(
                f"- Column '{column_name}': "
                f"expected {expected_type.simpleString()}, "
                f"found {actual_type.simpleString()}"
            )

    if validation_errors:
        error_report = (
            "schema validation failed.\n\n"
            + "\n".join(validation_errors)
        )

    raise ValueError(error_report)

In [ ]:
from typing import Sequence
def validate_allowed_values(
        df: DataFrame,
        column_name: str,
        allowed_values: Sequence[str]
)-> None:
    """
    Validate that all non-values in a column belong to a predefined list of allowed values

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    column_name: str 
        name of column to validate

    allowed_values: Sequence[str]
        List of allowed values in the column

    
    Raises
    ------
    ValueError
        If the column does not exist or contains one or more values outside the allowed_values list

    
    Return
    ------
    None
        ReturnsNone if validation passes.    
    
    """

    #check if column exist
    if column_name not in df.columns:
        raise ValueError(
            f" - Missing column '{column_name}' does not exist"
        )
    
    # Filter rows where value is not NULL
    invalid_rows = df.filter(
        col(column_name).isNotNull() 
        & (~col(column_name).isin(allowed_values))
    )

    # count invalid rows
    invalid_count = invalid_rows.count()

    # validation failed
    if invalid_count > 0:
        # print(f"Sample invalid calues found in '{column_name}':")

        invalid_values = [
            row[column_name]
            for row in invalid_rows
            .select(column_name)
            .distinct()
            .collect() # I used collect Because this function is not responsible for displaying data.
        ]


        raise ValueError(
            f"Validation failed: Column '{column_name}' "
            f"Found {invalid_count} invalid value(s). "
            f"Invalid values: {invalid_values}."
            f"Allowed values are: {list(allowed_values)}"
            
        )

    


In [ ]:
def validate_unique_keys(
        df: DataFrame,
        column_name: str
)-> None:

    """
    Parameters
    ----------
    df: DataFrame
        input Spark DataFrame
    
    column_name: str
        Name of the column to validate

    
    Raises
    ------
    ValueError
        if the column does not exist or Duplicate values are detected

        
    Returns
    -------
    None
        Returns None when all values are unique.
    """

    if column_name not in df.columns:
        raise ValueError(
            f"- Missing column '{column_name}' does not exist"
        )


    duplicate_keys = (
        df.filter(col(column_name).isNotNull())
        .groupBy(column_name)
        .count()
        .filter(col("count") > 1)
        .orderBy(col("count").desc())
    )


    duplicate_count = duplicate_keys.count()

    if duplicate_count > 0:        

        sample_duplicates = duplicate_keys.limit(5).collect()

        sample_report = ", ".join(
            f"{row[column_name]} ({row['count']} occurences)"
            for row in sample_duplicates
        )


        raise ValueError(
            f"Validation Failed: Found {duplicate_count} duplicated "
            
            f"key(s) in column '{column_name}'. "
            f"sample duplicate keys: {sample_report}"
        )

In [ ]:
def validate_numeric_ranges(
    df: DataFrame,
    column_name: str,
    minimum: float, # we use float to keep it reusable
    maximum: float
) -> None:
    """
        Validate that all non-null numeric values in a column fall
    within a specified inclusive range.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame.

    column_name : str
        Name of the numeric column to validate.

    minimum : float
        Minimum acceptable value (inclusive).

    maximum : float
        Maximum acceptable value (inclusive).

    Raises
    ------
    ValueError
        If the column does not exist or contains values outside
        the specified numeric range.

    Returns
    -------
    None
        Returns None if validation passes.
    """

    if column_name not in df.columns:
        raise ValueError(
            f" - Missing Column '{column_name}' does not exist"
        )
    if minimum > maximum:
        raise ValueError(
            "Minimum cannot be greater than maximum"
        )

    invalid_rows = (
        df.filter(
            col(column_name).isNotNull()
            &
            (
                (col(column_name) < minimum)
                |
                (col(column_name) > maximum)
            )
        )
    )

    invalid_count = invalid_rows.count()

    if invalid_count > 0:

        sample_invalid_values = [
            row[column_name]
            for row in invalid_rows
                .select(column_name)
                .distinct()
                .limit()
                .collect()
        ]

        raise ValueError(
            f"Validate failed: Found {invalid_count} value(s) "
            f"outside the allowed range "
            f"[{minimum}, {maximum}] "
            f"Sample values: {sample_invalid_values}"
        )



# Data Normalisation | Data Modeling

In [ ]:
# Creating the Customer Dimesion

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def create_customer_dimension(
    df: DataFrame,
    customer_columns: list[str]
)->DataFrame:

    """
    Parameters
    ---------
    df: DataFrame
        input Spark DataFrame

    customer_columns
        list of customer columns used to construct the customer dimension

    Raises
    -----
    ValueError
        if column one or more cuctomer columns do not exist in the DataFrame

    Returns
    ------
        Spark DataFrame containing unique customer records with a generated 
        customer_id surrogate key

    """
    # validate required columns
    validate_required_columns(  # we have previously written this function, so we are calling it here.
        df, customer_columns
    )

    # select only customer columns
    customers_df = df.select(*customer_columns)

    # Remove exact duplicate customer records
    customers_df = customers_df.dropDuplicates(customer_columns)

    # Generating the surrogate Key
    window_spec = Window.orderBy(
        "customer_name", 
        "email"
    )
    customers_df = customers_df.withColumn( # -> withColumn() adds a new column 
        "customer_id",
        row_number().over(window_spec) # -> row_number() generates sequential numbers | .over() applies the ordering rules
    )

    return customers_df




In [ ]:
# Create fact dimension df

from functools import reduce


def create_transactions_fact(
    transactions_df: DataFrame,
    customers_df: DataFrame,
    fact_columns: list[str],
    join_columns: list[str]
) -> DataFrame:
    """
    Create a transaction fact table by joining the transaction
    DataFrame with the customer dimension.

    Parameters
    ----------
    transactions_df : DataFrame
        Cleaned transaction DataFrame.

    customers_df : DataFrame
        Customer dimension containing the surrogate customer_id.

    fact_columns : list[str]
        Transaction columns to include in the fact table.

    join_columns : list[str]
        Columns used to join the transaction DataFrame with the
        customer dimension.

    Raises
    ------
    ValueError
        If one or more required columns are missing.

    Returns
    -------
    DataFrame
        Transaction fact table.
    """

    # ------------------------------------------
    # Validate required columns
    # ------------------------------------------

    validate_required_columns(
        transactions_df,
        fact_columns + join_columns
    )

    validate_required_columns(
        customers_df,
        ["customer_id"] + join_columns
    )

    # ------------------------------------------
    # Create aliases
    # ------------------------------------------

    t = transactions_df.alias("t")
    c = customers_df.alias("c")

    # ------------------------------------------
    # Build the join condition dynamically
    # ------------------------------------------

    join_condition = reduce(
        lambda condition1, condition2: condition1 & condition2,
        [
            col(f"t.{column}") == col(f"c.{column}")
            for column in join_columns
        ]
    )

    # ------------------------------------------
    # Join transaction data with customer dimension
    # ------------------------------------------

    joined_df = t.join(
        c,
        join_condition,
        how="left"
    )

    # ------------------------------------------
    # Build the list of columns to select
    # ------------------------------------------

    selected_columns = [
        col("c.customer_id")
    ]

    selected_columns.extend(
        col(f"t.{column}")
        for column in fact_columns
    )

    # ------------------------------------------
    # Create fact table
    # ------------------------------------------

    fact_transactions_df = joined_df.select(
        *selected_columns
    )

    return fact_transactions_df